# MSCI India Job Scraper
## India Jobs Scraper (v2.1 - March 2026)
**Source:** careers.msci.com

**ATS Detection:** Automatic API + Selenium fallback

In [1]:
!pip install selenium webdriver-manager pandas openpyxl requests beautifulsoup4 lxml playwright -q


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import sys, time, random, re
from pathlib import Path

# Add scripts dir to path so we can import scraper_utils
SCRIPTS_DIR = Path.home() / "Job_Scrapers" / "All_Scripts"
sys.path.insert(0, str(SCRIPTS_DIR))

from scraper_utils import *
import requests
from bs4 import BeautifulSoup
from datetime import datetime, timedelta

# ── LOCATION CONFIG ──────────────────────────────────────────────────────────
# Change to "" to scrape globally (all countries).
# The matching pipeline's pre-filter handles India-specific narrowing.
# Set to "India" here only if you want to reduce volume at scrape time.
LOCATION_FILTER = ""
COUNTRY_CODE   = ""  # e.g. "in" for SmartRecruiters country= param; "" = all
# ─────────────────────────────────────────────────────────────────────────────

print("Imports loaded. Date:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print(f"Location filter: '{LOCATION_FILTER}' (empty = broad/global scraping)")


scraper_utils.py loaded successfully
Schema: 25 columns
Skills DB: 79 skills
Imports loaded. Date: 2026-03-31 23:46:10
Location filter: '' (empty = broad/global scraping)


In [3]:
COMPANY = "MSCI"
OUTPUT_DIR = get_output_dir(COMPANY)
print(f"Output directory: {OUTPUT_DIR}")


Output directory: /Users/incognito/Job_Scrapers/All_CSV_Outputs/MSCI/Outputs/2026_03_31


In [4]:
print("=" * 60)
print("MSCI INDIA JOB SCRAPER")
print("Source: careers.msci.com — Custom portal (Selenium)")
print("=" * 60)

from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import json


def fetch_jd_selenium(driver, url, timeout=10):
    """Visit a job detail page and extract the JD text."""
    try:
        driver.get(url)
        time.sleep(random.uniform(2, 4))
        soup = BeautifulSoup(driver.page_source, "lxml")
        # Try common JD container selectors
        for sel in ["[class*='job-description']", "[class*='jd-info']", "[class*='description']",
                     "[class*='details']", "article", "main", ".content"]:
            el = soup.select_one(sel)
            if el and len(el.get_text(strip=True)) > 100:
                return el.get_text(" ", strip=True)
        # Fallback: get body text
        body = soup.select_one("body")
        return body.get_text(" ", strip=True)[:5000] if body else ""
    except Exception as e:
        print(f"    [WARN] JD fetch failed for {url}: {e}")
        return ""

def fetch_jd_requests(session, url):
    """Fetch a job detail page via requests and extract JD text."""
    try:
        resp = session.get(url, timeout=20)
        if resp.status_code == 200:
            soup = BeautifulSoup(resp.text, "lxml")
            for sel in ["[class*='job-description']", "[class*='jd-info']", "[class*='description']",
                         "[class*='details']", "article", "main"]:
                el = soup.select_one(sel)
                if el and len(el.get_text(strip=True)) > 100:
                    return el.get_text(" ", strip=True)
            body = soup.select_one("body")
            return body.get_text(" ", strip=True)[:5000] if body else ""
    except:
        pass
    return ""


msci_jobs = []

# Try API first — MSCI careers may have a hidden JSON endpoint
session = get_session()

API_ATTEMPTS = [
    ("https://careers.msci.com/api/jobs?location=India&limit=100", "GET"),
    ("https://careers.msci.com/api/search?q=&location=India", "GET"),
    ("https://careers.msci.com/search/jobs?location=India", "GET"),
]

for api_url, method in API_ATTEMPTS:
    try:
        print(f"  Trying API: {api_url}")
        resp = session.get(api_url, timeout=15)
        if resp.status_code == 200 and "application/json" in resp.headers.get("Content-Type", ""):
            data = resp.json()
            jobs_list = data.get("jobs", data.get("results", data.get("data", [])))
            if jobs_list:
                print(f"  API found {len(jobs_list)} jobs")
                for job in jobs_list:
                    loc = str(job.get("location", "India"))
                    msci_jobs.append({
                        "job_id": str(job.get("id", len(msci_jobs))),
                        "title": job.get("title", job.get("name", "")),
                        "company_name": "MSCI",
                        "job_url": job.get("url", job.get("apply_url", "")),
                        "business_unit": str(job.get("department", job.get("category", ""))),
                        "raw_jd_text": html_to_text(job.get("description", "")),
                        "location_city": loc.split(",")[0].strip(),
                        "location_country": "India",
                        "industry": "Financial Services / Investment Research / Analytics",
                        "date_posted": str(job.get("updated_at", datetime.now().strftime("%Y-%m-%d")))[:10],
                        "is_active": True,
                        "salary_currency": "INR",
                        "source_platform": "MSCI API",
                    })
                if msci_jobs:
                    break
    except Exception as e:
        print(f"  {api_url} failed: {e}")

# Selenium approach on careers.msci.com
if len(msci_jobs) < 5:
    print("\n  Trying Selenium on careers.msci.com...")
    driver = setup_selenium()
    try:
        driver.get("https://careers.msci.com/job-search")
        time.sleep(10)

        # Try to extract from SSR / page state
        try:
            next_data = driver.execute_script("return JSON.stringify(window.__REDUX_STATE__ || window.__INITIAL_STATE__ || window.__NEXT_DATA__ || {})")
            if next_data and len(next_data) > 200:
                print(f"  Found page state ({len(next_data)} chars)")
                nd = json.loads(next_data)
                # Look for job arrays in the state
                def extract_from_state(obj, depth=0):
                    if depth > 8: return []
                    if isinstance(obj, list) and len(obj) > 2:
                        if all(isinstance(i, dict) and any(k in i for k in ["title","jobTitle","name"]) for i in obj[:3]):
                            return obj
                    if isinstance(obj, dict):
                        for key in ["jobs","positions","listings","searchResults","results"]:
                            if key in obj:
                                r = extract_from_state(obj[key], depth+1)
                                if r: return r
                        for v in obj.values():
                            r = extract_from_state(v, depth+1)
                            if r: return r
                    return []
                jobs_raw = extract_from_state(nd)
                if jobs_raw:
                    print(f"  Found {len(jobs_raw)} jobs in page state")
                    india_kw = ["india", "bengaluru", "hyderabad", "pune", "mumbai", "delhi"]
                    for job in jobs_raw:
                        loc = str(job.get("location", job.get("office", job.get("city", "India"))))
                        if not any(k in loc.lower() for k in india_kw):
                            continue
                        msci_jobs.append({
                            "job_id": str(job.get("id", job.get("jobId", len(msci_jobs)))),
                            "title": job.get("title", job.get("jobTitle", job.get("name", ""))),
                            "company_name": "MSCI",
                            "job_url": job.get("url", job.get("applyUrl", "")),
                            "business_unit": str(job.get("department", job.get("team", ""))),
                            "raw_jd_text": html_to_text(job.get("description", "")),
                            "location_city": loc.split(",")[0].strip(),
                            "location_country": "India",
                            "industry": "Financial Services / Investment Research / Analytics",
                            "date_posted": datetime.now().strftime("%Y-%m-%d"),
                            "is_active": True,
                            "salary_currency": "INR",
                            "source_platform": "MSCI SSR",
                        })
        except Exception as e:
            print(f"  State extraction failed: {e}")

        # DOM scraping
        if not msci_jobs:
            try:
                WebDriverWait(driver, 20).until(
                    EC.presence_of_element_located((By.CSS_SELECTOR,
                        "[class*='job'], [class*='position'], [class*='opportunity'], a[href*='/job/']"))
                )
            except:
                time.sleep(5)

            for scroll in range(15):
                driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
                time.sleep(2)

            soup = BeautifulSoup(driver.page_source, "lxml")
            india_kw = ["india", "bengaluru", "hyderabad", "pune", "mumbai",
                        "delhi", "gurugram", "noida", "chennai"]

            # Remove nav/header/footer
            for el in soup.select("nav, header, footer, [class*='nav'], [class*='header']"):
                el.decompose()

            cards = (soup.select("[class*='job-card'], [class*='JobCard'], [class*='position-card']") or
                     soup.select("a[href*='/job/'], a[href*='/careers/job']"))

            for card in cards:
                title_el = card.select_one("h2, h3, h4, [class*='title'], a")
                title = title_el.get_text(strip=True) if title_el else ""
                href = card.get("href", "") if card.name == "a" else ""
                if not href:
                    link = card.select_one("a[href]")
                    href = link.get("href", "") if link else ""

                loc_el = card.select_one("[class*='location'], [class*='office'], [class*='city']")
                loc = loc_el.get_text(strip=True) if loc_el else "India"

                if is_valid_job_title(title) and any(k in loc.lower() for k in india_kw):
                    full_url = href if href.startswith("http") else f"https://careers.msci.com{href}"
                    msci_jobs.append({
                        "job_id": href.split("/")[-1] if href else str(len(msci_jobs)),
                        "title": title,
                        "company_name": "MSCI",
                        "job_url": full_url,
                        "business_unit": "",
                        "raw_jd_text": card.get_text(" ", strip=True),
                        "location_city": loc.split(",")[0].strip(),
                        "location_country": "India",
                        "industry": "Financial Services / Investment Research / Analytics",
                        "date_posted": datetime.now().strftime("%Y-%m-%d"),
                        "is_active": True,
                        "salary_currency": "INR",
                        "source_platform": "MSCI Selenium",
                    })

    except Exception as e:
        print(f"  Selenium error: {e}")
        import traceback; traceback.print_exc()
    finally:
        driver.quit()

# Deduplicate by title
seen_titles = set()
msci_jobs_deduped = []
for j in msci_jobs:
    if j["title"] not in seen_titles:
        seen_titles.add(j["title"])
        msci_jobs_deduped.append(j)
msci_jobs = msci_jobs_deduped

print(f"Total MSCI India jobs: {len(msci_jobs)}")


MSCI INDIA JOB SCRAPER
Source: careers.msci.com — Custom portal (Selenium)
  Trying API: https://careers.msci.com/api/jobs?location=India&limit=100


  Trying API: https://careers.msci.com/api/search?q=&location=India


  Trying API: https://careers.msci.com/search/jobs?location=India



  Trying Selenium on careers.msci.com...


Total MSCI India jobs: 0


In [5]:
df_msci = save_results(msci_jobs, "MSCI", OUTPUT_DIR)
if df_msci is not None:
    print(f"\nSample jobs:")
    cols = ["title","location_city","seniority_level","business_unit","job_url"]
    cols = [c for c in cols if c in df_msci.columns]
    print(df_msci[cols].head(10).to_string())


  [WARN] No jobs found for MSCI
